In [ ]:
import argparse
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
warnings.filterwarnings("ignore")

EPS = 1e-9
SMOOTHING = 8.0

KEY_GROUPS = {
    "vp": ["viewport_width", "viewport_height"],
    "counts": ["mouse_events_total", "touch_events_total"],
    "vp_counts": ["viewport_width", "viewport_height", "mouse_events_total", "touch_events_total"],
    "vp_hover": ["viewport_width", "viewport_height", "hover_zero"],
}


In [ ]:
def safe_div(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.abs(b) > EPS, a / b, np.nan)


In [ ]:
def dist_xy(x1, y1, x2, y2):
    return np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)


In [ ]:
def is_missing_obj(x):
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


In [ ]:
def event_get(ev, key, default=np.nan):
    if ev is None:
        return default
    if isinstance(ev, dict):
        return ev.get(key, default)
    try:
        return getattr(ev, key)
    except Exception:
        pass
    return default


In [ ]:
def normalize_event_list(obj):
    """Return list-like event container as a Python list.

    Parquet nested columns may come back as list, numpy array, pandas object,
    or None depending on engine/version.
    """
    if obj is None:
        return []
    if isinstance(obj, float) and math.isnan(obj):
        return []
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, list):
        return obj
    if isinstance(obj, tuple):
        return list(obj)
    try:
        if pd.isna(obj):
            return []
    except Exception:
        pass
    try:
        return list(obj)
    except Exception:
        return []


In [ ]:
def extract_sequence_stats(events_col, prefix, df):
    rows = []

    for obj in events_col:
        events = normalize_event_list(obj)
        ts, xs, ys = [], [], []

        for ev in events:
            t = event_get(ev, "timestamp_", np.nan)
            x = event_get(ev, "x_", np.nan)
            y = event_get(ev, "y_", np.nan)
            ts.append(t)
            xs.append(x)
            ys.append(y)

        t = np.asarray(ts, dtype=float)
        x = np.asarray(xs, dtype=float)
        y = np.asarray(ys, dtype=float)

        valid = ~(np.isnan(t) | np.isnan(x) | np.isnan(y))
        t, x, y = t[valid], x[valid], y[valid]
        n = len(t)

        d = {}
        d[f"{prefix}_len"] = n

        if n == 0:
            base_names = [
                "dur", "x_rng", "y_rng", "x_std", "y_std", "unique_ratio",
                "x_first", "y_first", "x_last", "y_last", "t_first", "t_last",
                "path", "disp", "straight", "dt_mean", "dt_std", "dt_min", "dt_max",
                "step_mean", "step_std", "step_max", "speed_mean", "speed_std",
                "speed_max", "stationary_ratio"
            ]
            for name in base_names:
                d[f"{prefix}_{name}"] = np.nan
            rows.append(d)
            continue

        d[f"{prefix}_dur"] = t[-1] - t[0]
        d[f"{prefix}_x_rng"] = np.nanmax(x) - np.nanmin(x)
        d[f"{prefix}_y_rng"] = np.nanmax(y) - np.nanmin(y)
        d[f"{prefix}_x_std"] = np.nanstd(x)
        d[f"{prefix}_y_std"] = np.nanstd(y)
        d[f"{prefix}_unique_ratio"] = len(set(zip(x, y))) / max(n, 1)
        d[f"{prefix}_x_first"] = x[0]
        d[f"{prefix}_y_first"] = y[0]
        d[f"{prefix}_x_last"] = x[-1]
        d[f"{prefix}_y_last"] = y[-1]
        d[f"{prefix}_t_first"] = t[0]
        d[f"{prefix}_t_last"] = t[-1]

        if n >= 2:
            dx = np.diff(x)
            dy = np.diff(y)
            dt = np.diff(t)
            step = np.sqrt(dx ** 2 + dy ** 2)
            positive_dt = np.where(np.abs(dt) > EPS, dt, np.nan)
            speed = step / positive_dt

            path = np.nansum(step)
            disp = math.sqrt((x[-1] - x[0]) ** 2 + (y[-1] - y[0]) ** 2)

            d[f"{prefix}_path"] = path
            d[f"{prefix}_disp"] = disp
            d[f"{prefix}_straight"] = disp / path if path > EPS else np.nan
            d[f"{prefix}_dt_mean"] = np.nanmean(dt)
            d[f"{prefix}_dt_std"] = np.nanstd(dt)
            d[f"{prefix}_dt_min"] = np.nanmin(dt)
            d[f"{prefix}_dt_max"] = np.nanmax(dt)
            d[f"{prefix}_step_mean"] = np.nanmean(step)
            d[f"{prefix}_step_std"] = np.nanstd(step)
            d[f"{prefix}_step_max"] = np.nanmax(step)
            d[f"{prefix}_speed_mean"] = np.nanmean(speed)
            d[f"{prefix}_speed_std"] = np.nanstd(speed)
            d[f"{prefix}_speed_max"] = np.nanmax(speed)
            d[f"{prefix}_stationary_ratio"] = np.nanmean(step == 0)
        else:
            for name in [
                "path", "disp", "straight", "dt_mean", "dt_std", "dt_min", "dt_max",
                "step_mean", "step_std", "step_max", "speed_mean", "speed_std",
                "speed_max", "stationary_ratio"
            ]:
                d[f"{prefix}_{name}"] = np.nan

        rows.append(d)

    seq = pd.DataFrame(rows, index=df.index)

    # Distances from first/last event point to click/hover points.
    for side in ["last", "first"]:
        x_col = f"{prefix}_x_{side}"
        y_col = f"{prefix}_y_{side}"
        seq[f"{prefix}_{side}_to_down"] = dist_xy(
            seq[x_col], seq[y_col], df["pointerdown_x"], df["pointerdown_y"]
        )
        seq[f"{prefix}_{side}_to_up"] = dist_xy(
            seq[x_col], seq[y_col], df["pointerup_x"], df["pointerup_y"]
        )

    seq[f"{prefix}_last_to_hover"] = dist_xy(
        seq[f"{prefix}_x_last"], seq[f"{prefix}_y_last"], df["hover_x"], df["hover_y"]
    )

    return seq


In [ ]:
def extract_base_features(df):
    out = pd.DataFrame(index=df.index)

    raw_cols = [
        "relative_captcha_init_time",
        "mouse_events_total",
        "touch_events_total",
        "pointerdown_timestamp",
        "pointerdown_x",
        "pointerdown_y",
        "pointerup_timestamp",
        "pointerup_x",
        "pointerup_y",
        "hover_timestamp",
        "hover_x",
        "hover_y",
        "viewport_width",
        "viewport_height",
    ]

    for col in raw_cols:
        if col in df.columns:
            out[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            out[col] = np.nan

    out["click_duration"] = out["pointerup_timestamp"] - out["pointerdown_timestamp"]
    out["time_init_down"] = out["pointerdown_timestamp"] - out["relative_captcha_init_time"]
    out["time_init_up"] = out["pointerup_timestamp"] - out["relative_captcha_init_time"]
    out["time_hover_down"] = out["pointerdown_timestamp"] - out["hover_timestamp"]
    out["time_hover_up"] = out["pointerup_timestamp"] - out["hover_timestamp"]

    out["pointer_dx"] = out["pointerup_x"] - out["pointerdown_x"]
    out["pointer_dy"] = out["pointerup_y"] - out["pointerdown_y"]
    out["pointer_dist"] = dist_xy(
        out["pointerdown_x"], out["pointerdown_y"], out["pointerup_x"], out["pointerup_y"]
    )
    out["click_speed"] = safe_div(out["pointer_dist"], out["click_duration"])

    out["hover_down_dist"] = dist_xy(
        out["hover_x"], out["hover_y"], out["pointerdown_x"], out["pointerdown_y"]
    )
    out["hover_up_dist"] = dist_xy(
        out["hover_x"], out["hover_y"], out["pointerup_x"], out["pointerup_y"]
    )

    out["hover_zero"] = (
        (out["hover_timestamp"] == 0) & (out["hover_x"] == 0) & (out["hover_y"] == 0)
    ).astype(int)

    out["viewport_area"] = out["viewport_width"] * out["viewport_height"]
    out["viewport_aspect"] = safe_div(out["viewport_width"], out["viewport_height"])
    out["viewport_invalid"] = ((out["viewport_width"] <= 0) | (out["viewport_height"] <= 0)).astype(int)

    out["vp_w_log"] = np.log1p(np.maximum(out["viewport_width"], 0))
    out["vp_h_log"] = np.log1p(np.maximum(out["viewport_height"], 0))
    out["vp_area_log"] = np.log1p(np.maximum(out["viewport_area"], 0))

    out["mobile_like"] = ((out["viewport_width"] > 0) & (out["viewport_width"] <= 600)).astype(int)
    out["desktop_like"] = ((out["viewport_width"] >= 1200) & (out["viewport_height"] >= 700)).astype(int)
    out["wide_screen"] = (out["viewport_width"] >= 1800).astype(int)

    out["pd_x_norm"] = safe_div(out["pointerdown_x"], out["viewport_width"])
    out["pd_y_norm"] = safe_div(out["pointerdown_y"], out["viewport_height"])
    out["pu_x_norm"] = safe_div(out["pointerup_x"], out["viewport_width"])
    out["pu_y_norm"] = safe_div(out["pointerup_y"], out["viewport_height"])
    out["h_x_norm"] = safe_div(out["hover_x"], out["viewport_width"])
    out["h_y_norm"] = safe_div(out["hover_y"], out["viewport_height"])

    out["pd_outside"] = (
        (out["pointerdown_x"] < 0) | (out["pointerdown_x"] > out["viewport_width"]) |
        (out["pointerdown_y"] < 0) | (out["pointerdown_y"] > out["viewport_height"])
    ).astype(int)
    out["pu_outside"] = (
        (out["pointerup_x"] < 0) | (out["pointerup_x"] > out["viewport_width"]) |
        (out["pointerup_y"] < 0) | (out["pointerup_y"] > out["viewport_height"])
    ).astype(int)
    out["h_outside"] = (
        (out["hover_x"] < 0) | (out["hover_x"] > out["viewport_width"]) |
        (out["hover_y"] < 0) | (out["hover_y"] > out["viewport_height"])
    ).astype(int)

    out["events_sum"] = out["mouse_events_total"] + out["touch_events_total"]
    out["mouse_touch_ratio"] = (out["mouse_events_total"] + 1) / (out["touch_events_total"] + 1)
    out["has_mouse"] = (out["mouse_events_total"] > 0).astype(int)
    out["has_touch"] = (out["touch_events_total"] > 0).astype(int)
    out["only_mouse"] = ((out["mouse_events_total"] > 0) & (out["touch_events_total"] == 0)).astype(int)
    out["only_touch"] = ((out["touch_events_total"] > 0) & (out["mouse_events_total"] == 0)).astype(int)

    mouse_lens = df["mouse_events"].apply(lambda x: len(normalize_event_list(x))) if "mouse_events" in df.columns else 0
    touch_lens = df["touch_events"].apply(lambda x: len(normalize_event_list(x))) if "touch_events" in df.columns else 0
    out["mouse_total_minus_len"] = out["mouse_events_total"] - mouse_lens
    out["touch_total_minus_len"] = out["touch_events_total"] - touch_lens
    out["mouse_truncated"] = (out["mouse_events_total"] > mouse_lens).astype(int)
    out["touch_truncated"] = (out["touch_events_total"] > touch_lens).astype(int)

    if "mouse_events" in df.columns:
        out = pd.concat([out, extract_sequence_stats(df["mouse_events"], "mouse", out)], axis=1)
    if "touch_events" in df.columns:
        out = pd.concat([out, extract_sequence_stats(df["touch_events"], "touch", out)], axis=1)

    # Replace infinite values with NaN so the imputer handles them.
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


In [ ]:
def make_keys(X, cols):
    # Convert rows into stable tuple keys. NaN is normalized to string sentinel.
    tmp = X[cols].copy()
    tmp = tmp.where(~tmp.isna(), "__NA__")
    return list(map(tuple, tmp.astype(str).values))


In [ ]:
def fit_te_maps(X, y, key_groups=KEY_GROUPS, smoothing=SMOOTHING):
    y = np.asarray(y, dtype=float)
    global_rate = float(np.mean(y))
    maps = {"global_rate": global_rate, "groups": {}}

    for name, cols in key_groups.items():
        keys = make_keys(X, cols)
        stat = defaultdict(lambda: [0.0, 0.0])
        for k, target in zip(keys, y):
            stat[k][0] += float(target)
            stat[k][1] += 1.0

        te_map = {}
        freq_map = {}
        for k, (bot_count, count) in stat.items():
            te_map[k] = (bot_count + global_rate * smoothing) / (count + smoothing)
            freq_map[k] = count

        maps["groups"][name] = {
            "cols": cols,
            "te": te_map,
            "freq": freq_map,
        }

    return maps


In [ ]:
def apply_te_maps(X, maps):
    X = X.copy()
    global_rate = maps["global_rate"]

    for name, cfg in maps["groups"].items():
        cols = cfg["cols"]
        keys = make_keys(X, cols)
        X[f"te_{name}"] = [cfg["te"].get(k, global_rate) for k in keys]
        X[f"freq_{name}"] = [cfg["freq"].get(k, 0.0) for k in keys]

    return X


In [ ]:
def get_models():
    return [
        (
            "logit",
            make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler(),
                LogisticRegression(C=0.2, max_iter=3000)
            ),
        ),
        (
            "extra_trees",
            make_pipeline(
                SimpleImputer(strategy="median"),
                ExtraTreesClassifier(
                    n_estimators=450,
                    max_features="sqrt",
                    min_samples_leaf=3,
                    random_state=1,
                    n_jobs=-1,
                )
            ),
        ),
        (
            "random_forest",
            make_pipeline(
                SimpleImputer(strategy="median"),
                RandomForestClassifier(
                    n_estimators=300,
                    max_features="sqrt",
                    min_samples_leaf=5,
                    random_state=2,
                    n_jobs=-1,
                )
            ),
        ),
    ]


In [ ]:
def choose_usable_columns(X):
    usable = []
    for col in X.columns:
        s = X[col]
        if s.isna().all():
            continue
        if s.nunique(dropna=False) <= 1:
            continue
        usable.append(col)
    return usable


In [ ]:
def fit_pipeline(train_path):
    train = pd.read_parquet(train_path)
    if "target" not in train.columns:
        raise ValueError("Training file must contain target column")

    y = train["target"].astype(int).values
    base = extract_base_features(train)

    # Full TE only to determine full candidate columns and final training matrix.
    full_maps = fit_te_maps(base, y)
    X_full_candidate = apply_te_maps(base, full_maps)
    usable_cols = choose_usable_columns(X_full_candidate)

    models = get_models()
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof = {name: np.zeros(len(train), dtype=float) for name, _ in models}

    for fold, (tr_idx, val_idx) in enumerate(skf.split(base, y), 1):
        X_tr_base = base.iloc[tr_idx]
        X_val_base = base.iloc[val_idx]
        y_tr = y[tr_idx]

        fold_maps = fit_te_maps(X_tr_base, y_tr)
        X_tr = apply_te_maps(X_tr_base, fold_maps).reindex(columns=usable_cols)
        X_val = apply_te_maps(X_val_base, fold_maps).reindex(columns=usable_cols)

        for name, model in models:
            m = clone(model)
            m.fit(X_tr, y_tr)
            oof[name][val_idx] = m.predict_proba(X_val)[:, 1]

    model_scores = {}
    for name, preds in oof.items():
        p = np.clip(preds, 1e-6, 1 - 1e-6)
        model_scores[name] = {
            "auc": roc_auc_score(y, p),
            "logloss": log_loss(y, p),
            "brier": brier_score_loss(y, p),
        }

    inv = {name: 1.0 / (score["logloss"] ** 2) for name, score in model_scores.items()}
    total_inv = sum(inv.values())
    weights = {name: val / total_inv for name, val in inv.items()}

    raw_oof = np.zeros(len(train), dtype=float)
    for name in oof:
        raw_oof += weights[name] * oof[name]
    raw_oof = np.clip(raw_oof, 1e-6, 1 - 1e-6)

    logit_oof = np.log(raw_oof / (1 - raw_oof)).reshape(-1, 1)
    calibrator = LogisticRegression(C=1.0, max_iter=1000)
    calibrator.fit(logit_oof, y)

    calibrated_oof = calibrator.predict_proba(logit_oof)[:, 1]
    final_oof = np.clip(0.7 * calibrated_oof + 0.3 * raw_oof, 0.001, 0.999)

    print("OOF model scores:")
    for name, score in model_scores.items():
        print(name, score, "weight=", weights[name])

    print("Raw ensemble:", {
        "auc": roc_auc_score(y, raw_oof),
        "logloss": log_loss(y, raw_oof),
        "brier": brier_score_loss(y, raw_oof),
    })
    print("Final calibrated blend:", {
        "auc": roc_auc_score(y, final_oof),
        "logloss": log_loss(y, final_oof),
        "brier": brier_score_loss(y, final_oof),
    })

    # Train final models on full training set with TE maps fitted on full train.
    X_full = X_full_candidate.reindex(columns=usable_cols)
    fitted_models = []
    for name, model in models:
        m = clone(model)
        m.fit(X_full, y)
        fitted_models.append((name, m))

    return {
        "te_maps": full_maps,
        "usable_cols": usable_cols,
        "weights": weights,
        "models": fitted_models,
        "calibrator": calibrator,
    }


In [ ]:
def predict_batch(df, state):
    base = extract_base_features(df)
    X = apply_te_maps(base, state["te_maps"]).reindex(columns=state["usable_cols"])

    raw = np.zeros(len(df), dtype=float)
    for name, model in state["models"]:
        raw += state["weights"][name] * model.predict_proba(X)[:, 1]

    raw = np.clip(raw, 1e-6, 1 - 1e-6)
    logit = np.log(raw / (1 - raw)).reshape(-1, 1)
    cal = state["calibrator"].predict_proba(logit)[:, 1]
    final = np.clip(0.7 * cal + 0.3 * raw, 0.001, 0.999)
    return final


In [ ]:
def predict_test_to_csv(test_path, out_path, state, batch_size=5000):
    pf = pq.ParquetFile(test_path)
    row_offset = 0
    first = True

    for batch in pf.iter_batches(batch_size=batch_size):
        df = batch.to_pandas()
        probs = predict_batch(df, state)
        ids = np.arange(row_offset, row_offset + len(df))
        row_offset += len(df)

        out = pd.DataFrame({"id": ids, "probability": probs})
        out.to_csv(out_path, mode="w" if first else "a", index=False, header=first)
        first = False

    print(f"Wrote {row_offset} predictions to {out_path}")


In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", required=True)
    parser.add_argument("--test", required=True)
    parser.add_argument("--out", required=True)
    parser.add_argument("--batch-size", type=int, default=5000)
    args = parser.parse_args()

    state = fit_pipeline(args.train)
    predict_test_to_csv(args.test, args.out, state, batch_size=args.batch_size)


In [ ]:
if __name__ == "__main__":
    main()
